In [ ]:
%pip install -q python-dotenv openai openpyxl

In [72]:
# imports
import os
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [73]:
# constants
load_dotenv()

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
NEBIUS_BASE_URL = (
    "https://api.studio.nebius.ai/v1/"  # "https://api.tokenfactory.nebius.com/v1/"
)
RAG_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
EMBED_MODEL = ""
JUDGE_MODEL = ""

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print("Client ready ✓")

Client ready ✓


In [74]:
# dataset
df = (
    pd.read_json(
        "hf://datasets/PatronusAI/financebench/financebench_merged.jsonl", lines=True
    )
    .sort_values(by="financebench_id", ascending=True)
    .reset_index(drop=True)
)

# for each row i, replace the "doc_link" value with the url from "https://github.com/patronus-ai/financebench/tree/main/pdfs" such that the url is: "https://github.com/patronus-ai/financebench/tree/main/pdfs/{{doc_name}}"
df["doc_link"] = df["doc_name"].apply(
    lambda x: f"https://github.com/patronus-ai/financebench/tree/main/pdfs/{x}.pdf"
)

print("columns:", df.columns.tolist())
df.head(2)

columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_00005,Corning,CORNING_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Trade accounts receivable, net of doubtful acc...",OPEN_SOURCE,[{'evidence_text': 'Consolidated Balance Sheet...,Information Technology,10k,2022,https://github.com/patronus-ai/financebench/tr...
1,financebench_id_00070,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",Accounts receivable+Income tax receivable+Unbi...,OPEN_SOURCE,[{'evidence_text': 'American Water Works Compa...,Utilities,10k,2022,https://github.com/patronus-ai/financebench/tr...


---
## Task 1 - naive generation

In [ ]:
# answer the first 5 questions of each question_type - 5 domain-relevant, 5 novel-generated

assignment2_naive_generation_filename = "assignment2_naive_generation.xlsx"


if os.path.exists(assignment2_naive_generation_filename):
    print("Loading existing results...")
    answers_df = pd.read_excel(assignment2_naive_generation_filename)
else:
    # Select the first 5 questions for each question_type
    questions_domain = df[df["question_type"] == "domain-relevant"].head(5)
    questions_novel = df[df["question_type"] == "novel-generated"].head(5)
    selected_questions = pd.concat([questions_domain, questions_novel]).reset_index(
        drop=True
    )

    answers = []

    for idx, row in selected_questions.iterrows():
        prompt = f"""
Answer the question in 2-4 sentences.
If you don't know the answer, say "I don't know".
QUESTION: {row["question"]}
ANSWER:
"""
        response = client.chat.completions.create(
            model=RAG_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=200,
        )
        naive_answer = response.choices[0].message.content.strip()
        result = pd.Series(
            {
                "financebench_id": row["financebench_id"],
                "question_type": row["question_type"],
                "question": row["question"],
                "naive_answer": naive_answer,
                "ground_truth": row["answer"],
                "verdict": "",  # correct/partially correct/wrong/refused
            }
        )
        answers.append(result)
        time.sleep(1.5)  # rate limiting

    answers_df = pd.DataFrame(answers)

    # save results before setting the verdict
    answers_df.to_excel(assignment2_naive_generation_filename, index=False)

In [110]:
# set verdict
VERDICT_MAP = {
    1: "correct",
    2: "partially correct",
    3: "wrong",
    4: "refused",
}
verdicts_dict = {
    "financebench_id_00005": VERDICT_MAP[1],
    "financebench_id_00070": VERDICT_MAP[4],
    "financebench_id_00080": VERDICT_MAP[1],
    "financebench_id_00206": VERDICT_MAP[1],
    "financebench_id_00215": VERDICT_MAP[2],
    "financebench_id_00283": VERDICT_MAP[3],
    "financebench_id_00288": VERDICT_MAP[4],
    "financebench_id_00299": VERDICT_MAP[4],
    "financebench_id_00302": VERDICT_MAP[4],
    "financebench_id_00382": VERDICT_MAP[2],
}

for fid, verdict in verdicts_dict.items():
    answers_df.loc[answers_df["financebench_id"] == fid, "verdict"] = verdict

answers_df.to_excel(assignment2_naive_generation_filename, index=False)

In [112]:
answers_df

,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company ha...",Yes. Corning had a positive working capital am...,correct
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,I don't know the specific details of American ...,"No, American Water Works had negative working ...",refused
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,"Based on FY2022 data, Paypal's working capital...",Yes. Paypal has a positive working capital of ...,correct
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,JPMorgan Chase's (JPM) gross margins are not a...,"Since JPM is a financial institution, gross ma...",correct
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,"Based on FY 2022 data, Verizon is considered a...",Yes. Verizon's capital intensity ratio was app...,partially correct
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,Pfizer expects to pay approximately $12 billio...,77.78,wrong
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,I don't know the specific details regarding th...,"Yes, there was a decline of ~42% between FY202...",refused
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,I don't know the specific details of JPM's bus...,Corporate. Its net revenue was -$473 million.,refused
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,I don't know the specific details of Pfizer's ...,"Yes, change in PPNE was positive year over year",refused
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,The region with the highest EBITDAR contributi...,Las Vegas resorts contributed ~90% of company ...,partially correct


#### Questions:

1. Cases where the model refused or asked for more information - why?
- In `novel-generated` Q.s, the model refused to answer (not enough knowledge) 3 times as opposed to only once for the given `domain-relevant` Q.s.<br>
I don't see a reason why sometimes it refuses to answer while sometime it hallucinates some answer...<br>
However, it DOES refuse because we told it in the prompt ('If you don't know the answer, say "I don't know"') - and it really does not have any relevant information for any of these questions.

2. Cases where the model answered confidently - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
- Overall, the model is confident in it's answers - regardless of the accuracy. Even when it refuses to answer - it explains why it cannot answer. 
For example, it hallucinates numbers when giving answers ("$12 billion") as if this number was derived from somewhere...
This is not surprising because that is the behaviour I encounter since starting using LLMs a couple of years ago - they always answer regardless of the data they have, mostly with a concrete answer, even when completely wrong. This holds also for the best models today.

3. Are there patterns by question_type? Do some types fail more than others?
- The `domain-relevant` Q.s are more yes/no Q.s.<br>
However, with the naive answers we received - I see no significant separation for the model's answers by the questions types...<br>
The one time it was completely wrong was when we asked it for a specific number ("How much ... **in USD million?**") - it just output some random number and was therefore wrong. For other questions, it was more like 50-50% ("Does Corning have positive working capital?") - and its best chance to succeed is to just say "yes" or "no". But it's worth as guessing.

---
## Task 2 - RAG reminder